# Chapter 02: How Actions Works: Event, Workflow, Job, Step, Runner (notebook edition)

## Learning Objectives

- Derive waves from needs
- Compute wall-clock from timestamps
- Relate wall-clock to billed minutes

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Load the real run

We read the fixture saved from run 37311041878. Set `GHA_MODE=live` to refetch it with `gh`. You should see three jobs.

In [ ]:
import json
from datetime import datetime
run = json.loads((ROOT / "fixtures" / "run_ch02_anatomy.json").read_text())
FMT = "%Y-%m-%dT%H:%M:%SZ"
t = {j["name"]: (datetime.strptime(j["started_at"], FMT), datetime.strptime(j["completed_at"], FMT)) for j in run["jobs"]}
print(sorted(t))
assert sorted(t) == ["build", "lint", "test"]

## 2. Durations and overlap

We compute each job's duration and confirm lint and test overlapped while build waited.

In [ ]:
dur = {k: int((e - s).total_seconds()) for k, (s, e) in t.items()}
print(dur)
assert dur == {"lint": 2, "test": 4, "build": 3}
assert t["lint"][0] < t["test"][1] and t["test"][0] < t["lint"][1]   # overlap
assert t["build"][0] >= max(t["lint"][1], t["test"][1])              # needs honored

## 3. Waves, wall-clock and billing

`needs` defines waves; billing rounds each job up to a minute.

In [ ]:
from intro_gha.workflow import execution_waves
from intro_gha.cost import run_minutes
print(execution_waves({"lint": [], "test": [], "build": ["lint", "test"]}))
wall = max(e for _, e in t.values()) - min(s for s, _ in t.values())
print("wall-clock s:", wall.seconds, "| billed minutes:", run_minutes([(d, "linux") for d in dur.values()]))
assert wall.seconds == 10 and run_minutes([(d, "linux") for d in dur.values()]) == 3

## Takeaways & Next Steps

- Jobs overlap unless `needs` says otherwise; `needs` waits for the slowest dependency.
- 10 seconds of wall-clock still bills 3 minutes.
- Next: Chapter 03 writes a first real workflow.

## Chapter Link

Read: `learning_modules/chapter_02_how_actions_works.md`